In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import shap

file_path = r"C:\Users\brend\OneDrive - Stonehill College\Swing_Data.xlsx"
df = pd.read_excel(file_path)

In [13]:
df['side_encoded'] = df['side'].map({'L': 0, 'R': 1})

df.rename(columns={
    'avg_bat_speed': 'bat_speed',
    'swing_tilt': 'vertical_tilt',
    'avg_swing_length': 'swing_length',
    'avg_intercept_y_vs_batter': 'contact_point_rel_y'
}, inplace=True)

In [14]:
df['attack_direction_mirrored'] = df['attack_direction']
df.loc[df['side_encoded'] == 1, 'attack_direction_mirrored'] *= -1
df.loc[df['side_encoded'] == 0, 'attack_direction_mirrored'] *= -1
df['norm_attack_direction_mirrored'] = (
    df['attack_direction_mirrored'] / df['attack_direction_mirrored'].abs().max()
)

df['tilt_angle_interaction'] = df['vertical_tilt'] * df['attack_angle']
df['vertical_reach'] = df['attack_angle'] - df['contact_point_rel_y']
df['tilt_length_ratio'] = df['vertical_tilt'] / df['swing_length']
df['attack_angle_abs'] = df['attack_angle'].abs()
df['direction_consistency'] = df['attack_direction_mirrored'] ** 2
df['tilt_attack_ratio'] = df['vertical_tilt'] / (df['attack_angle'] + 0.001)

In [15]:
feature_cols = [
    'bat_speed', 'vertical_tilt', 'attack_angle', 'attack_direction_mirrored',
    'swing_length', 'contact_point_rel_y', 'tilt_angle_interaction',
    'norm_attack_direction_mirrored', 'vertical_reach', 'tilt_length_ratio',
    'attack_angle_abs', 'direction_consistency', 'tilt_attack_ratio',
    'side_encoded'
]

X = df[feature_cols]

y = df['Contact% (sc)']

In [16]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

preds = np.zeros(len(df))
models = []
feature_importances = []
shap_interactions_list = []

for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train = y.iloc[train_idx]
    weights_train = df.iloc[train_idx]['competitive_swings']

    model = XGBRegressor(
        n_estimators=1000,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train, sample_weight=weights_train)
    preds[test_idx] = model.predict(X_test)

    models.append(model)
    feature_importances.append(model.feature_importances_)

    explainer = shap.TreeExplainer(model)
    shap_interactions = explainer.shap_interaction_values(X_test)
    shap_interactions_list.append(shap_interactions.mean(axis=0))

df['pContact_model_oof'] = preds

In [17]:
side_params = {}

for side in [0, 1]:
    mask = df['side_encoded'] == side
    mean_pred = preds[mask].mean()
    std_pred = preds[mask].std()

    side_params[side] = {
        "mean": mean_pred,
        "std": std_pred
    }

    df.loc[mask, 'cSwing+'] = (
        100 + 10 * (preds[mask] - mean_pred) / std_pred
    )

In [18]:
df['pContact_scaled'] = np.nan

for side in [0, 1]:
    mask = df['side_encoded'] == side
    mean_pred = side_params[side]["mean"]
    std_pred = side_params[side]["std"]

    df.loc[mask, 'pContact_scaled'] = (
        mean_pred +
        (df.loc[mask, 'cSwing+'] - 100) / 10 * std_pred
    )

df['pContact_scaled'] = df['pContact_scaled'].round(3)

In [19]:
output_file = r"C:\Users\brend\OneDrive - Stonehill College\swing_plus_contact_results.xlsx"
df_output = df[
    ['year', 'Team', 'name', 'cSwing+', 'Contact% (sc)', 'pContact_scaled']
]
df_output.to_excel(output_file, index=False)

print(f"cSwing+ calculations complete! Results saved to {output_file}")

cSwing+ calculations complete! Results saved to C:\Users\brend\OneDrive - Stonehill College\swing_plus_contact_results.xlsx


In [20]:
teams = ['BOS', 'NYM', 'NYY']
df_2025 = df[(df['year'] == 2025) & (df['Team'].isin(teams))]

export_cols = ['year', 'Team', 'name', 'cSwing+'] + feature_cols
top_bottom_list = []

for team in teams:
    team_df = df_2025[df_2025['Team'] == team].copy()

    top3 = team_df.nlargest(3, 'cSwing+')
    bottom3 = team_df.nsmallest(3, 'cSwing+')

    top_bottom_list.append(top3)
    top_bottom_list.append(bottom3)

result_df = pd.concat(top_bottom_list)[export_cols]

output_file = r"C:\Users\brend\OneDrive - Stonehill College\swing_plus_top_bottom_2025_contact_features.xlsx"
result_df.to_excel(output_file, index=False)

print(f"Top and bottom 3 cSwing+ players for BOS, NYM, NYY in 2025 exported to {output_file}")

Top and bottom 3 cSwing+ players for BOS, NYM, NYY in 2025 exported to C:\Users\brend\OneDrive - Stonehill College\swing_plus_top_bottom_2025_contact_features.xlsx


In [21]:
def recompute_engineered_features(row):
    row['tilt_angle_interaction'] = row['vertical_tilt'] * row['attack_angle']
    row['swing_aggressiveness'] = row['bat_speed'] * row['swing_length']
    row['vertical_reach'] = row['attack_angle'] - row['contact_point_rel_y']
    row['tilt_length_ratio'] = row['vertical_tilt'] / row['swing_length']
    return row

def pcontact_to_cSwing(pcontact, side, side_params):
    mean_pred = side_params[side]["mean"]
    std_pred = side_params[side]["std"]
    return 100 + 10 * (pcontact - mean_pred) / std_pred


def simulate_metric_change_ensemble_cSwing(
    df,
    year,
    player_name,
    metric_changes,
    feature_cols,
    scaler,
    models,
    side_params
):

    player_mask = (df['year'] == year) & (df['name'] == player_name)
    if player_mask.sum() == 0:
        raise ValueError("Player not found for that year.")

    player = df.loc[player_mask].iloc[0].copy()

    player = recompute_engineered_features(player)

    X_base = player[feature_cols].values.reshape(1, -1)
    X_base_scaled = scaler.transform(X_base)

    base_preds = np.array([
        model.predict(X_base_scaled)[0] for model in models
    ])

    baseline_pContact = base_preds.mean()

    baseline_cSwing = pcontact_to_cSwing(
        baseline_pContact,
        player['side_encoded'],
        side_params
    )

    for metric, delta in metric_changes.items():
        if metric not in player:
            raise ValueError(f"Metric '{metric}' not found.")
        player[metric] += delta

    player = recompute_engineered_features(player)

    X_new = player[feature_cols].values.reshape(1, -1)
    X_new_scaled = scaler.transform(X_new)

    new_preds = np.array([
        model.predict(X_new_scaled)[0] for model in models
    ])

    new_pContact = new_preds.mean()

    side_std = side_params[player['side_encoded']]['std']
    delta_cSwing = 10 * (new_pContact - baseline_pContact) / side_std
    new_cSwing = baseline_cSwing + delta_cSwing

    result_table = pd.DataFrame([{
        "Year": year,
        "Player": player_name,
        "Baseline Contact% (model)": round(baseline_pContact, 3),
        "New Contact% (model)": round(new_pContact, 3),
        "Δ Contact% (model)": round(new_pContact - baseline_pContact, 4),
        "Baseline cSwing+": round(baseline_cSwing, 1),
        "New cSwing+": round(new_cSwing, 1),
        "Δ cSwing+": round(delta_cSwing, 2),
        "Metric Changes": metric_changes
    }])

    return result_table

In [22]:
result_table = simulate_metric_change_ensemble_cSwing(
    df=df,
    year=2025,
    player_name="Mayer, Marcelo",
    metric_changes={
        'bat_speed': 1
    },
    feature_cols=feature_cols,
    scaler=scaler,
    models=models,
    side_params=side_params
)

print(result_table)

   Year          Player  Baseline Contact% (model)  New Contact% (model)  \
0  2025  Mayer, Marcelo                      0.738                 0.739   

   Δ Contact% (model)  Baseline cSwing+  New cSwing+  Δ cSwing+  \
0               0.001              94.4         94.6        0.2   

     Metric Changes  
0  {'bat_speed': 1}  


C:\Users\brend\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\brend\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
